## Imports

imports do projeto

In [ ]:
import ctypes
import gc
import os
import subprocess
import time

from decimal import Decimal

import numpy as np
import pandas as pd
from PIL import Image

from models.avaliacao import Avaliacao
from models.indice_linha import IndiceLinha
from rembg import new_session, remove

# Define providers do ONNX Runtime conforme a máquina atual
import onnxruntime as ort

available_providers = ort.get_available_providers()

def nvidia_gpu_ativa():
    try:
        result = subprocess.run(
            ['nvidia-smi', '-L'],
            check=False,
            capture_output=True,
            text=True
        )
    except FileNotFoundError:
        return False

    return result.returncode == 0 and bool(result.stdout.strip())

def cuda_libs_disponiveis():
    try:
        ctypes.CDLL('libcublasLt.so.12')
        return True
    except OSError:
        return False

use_cuda = (
    'CUDAExecutionProvider' in available_providers
    and nvidia_gpu_ativa()
    and cuda_libs_disponiveis()
)

if use_cuda:
    ORT_PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    print('GPU NVIDIA ativa: usando CUDAExecutionProvider com fallback para CPU.')
else:
    ORT_PROVIDERS = ['CPUExecutionProvider']
    print('GPU NVIDIA/CUDA indisponivel: usando somente CPUExecutionProvider.')

print(f'Providers disponiveis no ambiente: {available_providers}')
print(f'Providers configurados para a sessao: {ORT_PROVIDERS}')

def resolver_providers(provider_config: str, nome_modelo: str) -> list[str]:
    """
    Retorna a lista de providers do ONNX Runtime.

    - 'cpu': sempre CPU
    - 'gpu': usa GPU se disponível, senão CPU com aviso
    """
    if provider_config == 'cpu':
        return ['CPUExecutionProvider']

    if provider_config == 'gpu':
        if use_cuda:
            return ['CUDAExecutionProvider', 'CPUExecutionProvider']
        else:
            print(f'[AVISO] Modelo {nome_modelo} configurado para GPU, mas GPU nao disponivel. Usando CPU.')
            return ['CPUExecutionProvider']

    raise ValueError(f"Provider '{provider_config}' invalido. Use 'gpu' ou 'cpu'.")


## Configuração

In [ ]:
# Diretórios
BASE_DIR = os.getcwd()

# Diretórios mais gerais
DATA_DIR = os.path.join(BASE_DIR, 'data')
GENERATED_DIR = os.path.join(BASE_DIR, 'generated')

# Diretórios específicos
ORIGINAL_PHOTOS_DIR = os.path.join(DATA_DIR, 'Original')
GROUND_OF_TRUTH = os.path.join(DATA_DIR, 'Ground-of-truth')

SEGMENTED_PHOTOS_DIR = os.path.join(GENERATED_DIR, 'Segmentada')
GROUND_OF_TRUTH_OUTPUT = os.path.join(GENERATED_DIR, 'Ground-of-truth')

# Caminhos de arquivos
INDICE_PATH = os.path.join(DATA_DIR, 'Indice.xlsx')

# Nomes das colunas do excel
NOME_COL = 'nome do arquivo'
FAZENDA_COL = 'fazenda'
PESO_COL = 'peso'

# Configuração do tipo de arquivo
ORIGINAL_IMAGE_TYPE = '.jpg'
REMBG_IMAGE_TYPE = '.png'

# Configuração de binarização
LIMIAR_BINARIZACAO = 127

# Modelos para avaliação (modelo: provider)
# provider: 'gpu' = usa GPU se disponível, senão CPU | 'cpu' = sempre CPU
MODELOS_PARA_AVALIACAO = {
    'u2net': 'gpu',
    'u2netp': 'gpu',
    'u2net_human_seg': 'gpu',
    # 'u2net_cloth_seg': 'gpu',  # não vai entrar porque é específico para roupas e altera a resolução da imagem
    'silueta': 'gpu',
    'isnet-general-use': 'gpu',
    'isnet-anime': 'gpu',
    'sam': 'gpu',
    'birefnet-general': 'cpu',
    'birefnet-general-lite': 'gpu',
    'birefnet-portrait': 'gpu',
    'birefnet-dis': 'gpu',
    'birefnet-hrsod': 'gpu',
    'birefnet-cod': 'gpu',
    'birefnet-massive': 'cpu',
}

## Leitura do índice


In [ ]:
indice_df = pd.read_excel(INDICE_PATH)
colunas = {c.strip().lower(): c for c in indice_df.columns}

nome_col = colunas.get(NOME_COL)
fazenda_col = colunas.get(FAZENDA_COL)
peso_col = colunas.get(PESO_COL)

if not nome_col or not fazenda_col or not peso_col:
    raise ValueError('Alguma das colunas esperadas não foi encontrada no arquivo Excel.')

indice_excel = [
    IndiceLinha(
        nome_arquivo=str(row[nome_col]),
        fazenda=str(row[fazenda_col]),
        peso=Decimal(str(row[peso_col]))
    )
    for _, row in indice_df.iterrows()
]

## Verificação de integridade dos PNGs gerados

Esta célula valida todos os arquivos .png dentro de `generated/` (incluindo subpastas). Se um arquivo estiver corrompido ou não puder ser aberto corretamente, ele é removido.

In [ ]:
# Verifica integridade dos PNGs em generated/ e remove arquivos corrompidos
def verificar_e_limpar_pngs_corrompidos(diretorio_base):
    total_png = 0
    arquivos_integros = 0
    arquivos_removidos = 0
    falhas_remocao = 0

    for raiz, _, arquivos in os.walk(diretorio_base):
        for nome_arquivo in arquivos:
            if not nome_arquivo.lower().endswith('.png'):
                continue

            total_png += 1
            caminho_arquivo = os.path.join(raiz, nome_arquivo)

            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify()

                with Image.open(caminho_arquivo) as img:
                    img.load()

                arquivos_integros += 1
            except Exception as e:
                print(f"Corrompido detectado: {caminho_arquivo}")
                print(f" - Erro: {e}")

                try:
                    os.remove(caminho_arquivo)
                    arquivos_removidos += 1
                    print(" - Arquivo removido com sucesso.")
                except OSError as erro_remocao:
                    falhas_remocao += 1
                    print(f" - Falha ao remover arquivo: {erro_remocao}")

    print('\nVerificacao de integridade concluida.')
    print(f" - Total de PNGs verificados: {total_png}")
    print(f" - Arquivos integros: {arquivos_integros}")
    print(f" - Arquivos removidos por corrupcao: {arquivos_removidos}")
    print(f" - Falhas ao remover: {falhas_remocao}")

verificar_e_limpar_pngs_corrompidos(GENERATED_DIR)

## Conversão do Ground Truth para PNG Binarizado

In [ ]:
# Cria o diretório de saída se não existir
os.makedirs(GROUND_OF_TRUTH_OUTPUT, exist_ok=True)

print(f"Convertendo máscaras do Ground Truth para PNG binarizado...")
print(f"Total de imagens: {len(indice_excel)}\n")

for idx, linha in enumerate(indice_excel, 1):
    # Caminho da máscara JPG original
    caminho_original = os.path.join(GROUND_OF_TRUTH, f"{linha.nome_arquivo}{ORIGINAL_IMAGE_TYPE}")

    # Verifica se o arquivo existe
    if not os.path.isfile(caminho_original):
        print(f"[{idx}/{len(indice_excel)}] AVISO: Máscara não encontrada: {caminho_original}")
        continue

    # Caminho de saída PNG
    caminho_saida = os.path.join(GROUND_OF_TRUTH_OUTPUT, f"{linha.nome_arquivo}{REMBG_IMAGE_TYPE}")

    # Verifica se já foi processado
    if os.path.isfile(caminho_saida):
        print(f"[{idx}/{len(indice_excel)}] Pulando (já convertido): {linha.nome_arquivo}")
        continue

    # Carrega a imagem JPG
    with Image.open(caminho_original) as img:
        # Converte para escala de cinza
        img_gray = img.convert('L')

        # Converte para array numpy
        matriz = np.array(img_gray)

        # Binariza: pixels > LIMIAR_BINARIZACAO → 255, senão → 0
        matriz_binarizada = np.where(matriz > LIMIAR_BINARIZACAO, 255, 0).astype(np.uint8)

        # Converte de volta para imagem PIL
        img_binarizada = Image.fromarray(matriz_binarizada, mode='L')

        # Salva como PNG
        img_binarizada.save(caminho_saida, format='PNG')

    print(f"[{idx}/{len(indice_excel)}] Convertido: {linha.nome_arquivo}")

print(f"\nConversão concluída! Máscaras salvas em: {GROUND_OF_TRUTH_OUTPUT}")

## Processa as imagens e gera a máscara binária

In [ ]:
def formatar_duracao(segundos):
    if segundos is None:
        return '--'

    total = int(max(0, segundos))
    horas = total // 3600
    minutos = (total % 3600) // 60
    seg = total % 60

    if horas > 0:
        return f'{horas}h{minutos:02d}m{seg:02d}s'
    if minutos > 0:
        return f'{minutos}m{seg:02d}s'
    return f'{seg}s'


def imprimir_status(geral, modelo, nome_modelo):
    processadas_geral = geral['processadas']
    total_geral = geral['total']

    perc_geral = (processadas_geral / total_geral * 100.0) if total_geral else 0.0
    tempo_total = time.perf_counter() - geral['inicio']
    taxa_geral = (processadas_geral / tempo_total) if tempo_total > 0 else 0.0

    processadas_modelo = modelo['processadas']
    total_modelo = modelo['total']
    perc_modelo = (processadas_modelo / total_modelo * 100.0) if total_modelo else 0.0
    media_modelo = (modelo['tempo_inferencia'] / modelo['ok']) if modelo['ok'] else 0.0

    tempo_modelo = time.perf_counter() - modelo['inicio']
    taxa_modelo = (processadas_modelo / tempo_modelo) if tempo_modelo > 0 else 0.0
    restantes_modelo = max(0, total_modelo - processadas_modelo)
    eta_modelo = (restantes_modelo / taxa_modelo) if taxa_modelo > 0 else None

    print(
        f"[GERAL ] {processadas_geral}/{total_geral} ({perc_geral:5.1f}%) | "
        f"ok={geral['ok']} skip={geral['skip']} erro={geral['erro']} | "
        f"{taxa_geral:.2f} img/s"
    )
    print(
        f"[MODELO {nome_modelo}] {processadas_modelo}/{total_modelo} ({perc_modelo:5.1f}%) | "
        f"ok={modelo['ok']} skip={modelo['skip']} erro={modelo['erro']} | "
        f"media={media_modelo:.2f}s/img | ETA {formatar_duracao(eta_modelo)}"
    )
    print()


total_modelos = len(MODELOS_PARA_AVALIACAO)
total_imagens = len(indice_excel)
total_previsto = total_modelos * total_imagens

stats_geral = {
    'total': total_previsto,
    'processadas': 0,
    'ok': 0,
    'skip': 0,
    'erro': 0,
    'inicio': time.perf_counter(),
    'tempo_inferencia': 0.0
}

for model, provider_config in MODELOS_PARA_AVALIACAO.items():
    providers = resolver_providers(provider_config, model)
    print(f'Iniciando modelo: {model} (provider: {provider_config})')

    stats_modelo = {
        'total': total_imagens,
        'processadas': 0,
        'ok': 0,
        'skip': 0,
        'erro': 0,
        'inicio': time.perf_counter(),
        'tempo_inferencia': 0.0
    }

    try:
        rembg_session = new_session(model, providers=providers)
    except Exception as e:
        print(f" - Falha ao iniciar sessao com providers {providers}: {e}")
        print(' - Recriando sessao em CPU...')
        rembg_session = new_session(model, providers=['CPUExecutionProvider'])

    output_dir = os.path.join(SEGMENTED_PHOTOS_DIR, model)
    os.makedirs(output_dir, exist_ok=True)

    for linha in indice_excel:
        original_path = os.path.join(ORIGINAL_PHOTOS_DIR, f"{linha.nome_arquivo}{ORIGINAL_IMAGE_TYPE}")
        mascara_path = os.path.join(GROUND_OF_TRUTH, f"{linha.nome_arquivo}{ORIGINAL_IMAGE_TYPE}")
        output_path = os.path.join(output_dir, f"{linha.nome_arquivo}{REMBG_IMAGE_TYPE}")

        if not os.path.isfile(original_path):
            print(f"[ERRO ] Arquivo original nao encontrado: {original_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            imprimir_status(stats_geral, stats_modelo, model)
            continue

        if not os.path.isfile(mascara_path):
            print(f"[ERRO ] Arquivo de mascara nao encontrado: {mascara_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            imprimir_status(stats_geral, stats_modelo, model)
            continue

        if os.path.isfile(output_path):
            stats_geral['processadas'] += 1
            stats_geral['skip'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['skip'] += 1
            imprimir_status(stats_geral, stats_modelo, model)
            continue

        try:
            inicio_inferencia = time.perf_counter()
            with Image.open(original_path) as input_rembg:
                output_rembg = remove(
                    input_rembg,
                    only_mask=True,
                    post_process_mask=True,
                    session=rembg_session
                )
            output_rembg.save(output_path)

            duracao_inferencia = time.perf_counter() - inicio_inferencia
            stats_geral['processadas'] += 1
            stats_geral['ok'] += 1
            stats_geral['tempo_inferencia'] += duracao_inferencia

            stats_modelo['processadas'] += 1
            stats_modelo['ok'] += 1
            stats_modelo['tempo_inferencia'] += duracao_inferencia

            imprimir_status(stats_geral, stats_modelo, model)
        except Exception as e:
            print(f"[ERRO ] Falha ao segmentar {linha.nome_arquivo} no modelo {model}: {e}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            imprimir_status(stats_geral, stats_modelo, model)

    tempo_modelo = time.perf_counter() - stats_modelo['inicio']
    taxa_modelo = (stats_modelo['processadas'] / tempo_modelo) if tempo_modelo > 0 else 0.0
    media_modelo = (stats_modelo['tempo_inferencia'] / stats_modelo['ok']) if stats_modelo['ok'] else 0.0

    print(f'[RESUMO MODELO {model}]')
    print(
        f"tempo_total={formatar_duracao(tempo_modelo)} | total={stats_modelo['total']} | "
        f"ok={stats_modelo['ok']} | skip={stats_modelo['skip']} | erro={stats_modelo['erro']}"
    )
    print(
        f"tempo_medio={media_modelo:.2f}s/img | throughput={taxa_modelo:.2f} img/s"
    )
    print('-' * 100)

    # Libera memoria da GPU entre modelos
    del rembg_session
    gc.collect()
    if use_cuda:
        # Força liberação de memória CUDA pelo ONNX Runtime
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
        except ImportError:
            pass  # torch não instalado, segue sem limpar cache CUDA

## Instancia as avaliações

In [ ]:
for linha in indice_excel:
    avaliacao = Avaliacao(linha.nome_arquivo, REMBG_IMAGE_TYPE, GROUND_OF_TRUTH_OUTPUT, SEGMENTED_PHOTOS_DIR,
                          list(MODELOS_PARA_AVALIACAO.keys()))

    print(f"A avaliação {avaliacao.nome_arquivo} teve como área {avaliacao.ground_truth.area} e perímetro {avaliacao.ground_truth.perimetro}")
    for model in avaliacao.segmetacoes:
        print(f"Avaliando modelo: {model.modelo}, área: {model.area}, perímetro: {model.perimetro}")